# 3. Topological Data Analysis (TDA)

**Objective**: Compute Persistent Homology to identify robust topological features (loops, voids) that persist across scales.

**Steps**:
1.  Load Preprocessed Data.
2.  **Persistent Homology**: Compute H0 (clusters) and H1 (loops) using Ripser.
3.  **Persistence Diagrams**: Visualize the birth and death of topological features.
4.  **3D Topology**: Analyze higher-order voids (H2) in the UMAP embedding.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

try:
    from ripser import ripser
    from persim import plot_diagrams
    print("ripser and persim libraries found.")
except ImportError:
    print("Installing ripser and persim libraries...")
    import sys
    !{sys.executable} -m pip install ripser persim
    from ripser import ripser
    from persim import plot_diagrams

# Import Architectural Style
import viz_style
viz_style.apply_style()

# Load Data
session_id = 715093703
input_file = Path(f"../Dataset/Processed/{session_id}/average_response_matrix.pkl")

if input_file.exists():
    average_response_matrix_total = pd.read_pickle(input_file)
    print(f"Loaded data from {input_file}")
else:
    print(f"Error: Data file not found at {input_file}")

### Step 2: Persistent Homology (H0, H1)
Compute persistent homology on the PCA-reduced data to find loops and connected components.

In [ ]:
# Prepare Data (PCA reduced)
X = average_response_matrix_total.values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_components_tda = 10
pca_tda = PCA(n_components=n_components_tda)
X_tda = pca_tda.fit_transform(X_scaled)

print(f"Data shape for TDA: {X_tda.shape} (PCA Top {n_components_tda} components)")

# Compute Persistent Homology
print("Computing Persistent Homology (maxdim=1)...")
result = ripser(X_tda, maxdim=1)
diagrams = result['dgms']

# Visualization
plt.figure(figsize=(8, 6))
plot_diagrams(diagrams, show=True)
plt.title(f"Persistence Diagram (PCA Top {n_components_tda})")
plt.show()

# Interpretation
print("--- Interpretation ---")
print("H0 (Blue): Connected Components. Points far from diagonal = distinct clusters.")
print("H1 (Orange): Loops/Cycles. Points far from diagonal = significant loops.")

### Step 3: 3D UMAP Topology (H2)
Analyze the topology of the 3D UMAP embedding to find voids (H2 features).

In [ ]:
# Compute 3D UMAP
print("Computing 3D UMAP for Topology Analysis...")
reducer_3d = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=3, random_state=42)
embedding_3d = reducer_3d.fit_transform(X_scaled)

# Compute Persistent Homology on 3D Embedding (up to H2)
print("Computing Persistent Homology on 3D UMAP (maxdim=2)...")
result_3d = ripser(embedding_3d, maxdim=2)
diagrams_3d = result_3d['dgms']

# Visualization
plt.figure(figsize=(10, 8))
plot_diagrams(diagrams_3d, show=True)
plt.title("Persistence Diagram of 3D UMAP Structure (H0, H1, H2)")
plt.show()

print("--- Interpretation ---")
print("H2 (Green): Voids/Cavities. Points far from diagonal = significant voids.")